# Fine-Tune DistilBERT for Sentiment Analysis
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ajit-ai/Data_Science/blob/main/05_NLP_Embeddings/sentiment_analysis_transformers.ipynb)

Fine-tuning adapts a pre-trained transformer to your labels. We train DistilBERT on the IMDB movie-review dataset using the `Trainer` API.

**Runtime tip:** enable GPU in Colab (*Runtime > Change runtime type > T4 GPU*) - training takes ~5 minutes on the subset used here.

In [ ]:
!pip install -q transformers datasets evaluate accelerate

## 1. Data: IMDB (small balanced subset)

In [ ]:
from datasets import load_dataset

ds = load_dataset("imdb")
small_train = ds["train"].shuffle(seed=42).select(range(2000))
small_test  = ds["test"].shuffle(seed=42).select(range(1000))
print(small_train[0]["text"][:200], "... ->", small_train[0]["label"])

## 2. Tokenize

In [ ]:
from transformers import AutoTokenizer

ckpt = "distilbert-base-uncased"
tok = AutoTokenizer.from_pretrained(ckpt)

def prep(batch):
    return tok(batch["text"], truncation=True, padding="max_length", max_length=256)

train_ds = small_train.map(prep, batched=True)
test_ds  = small_test.map(prep, batched=True)

## 3. Model + Trainer

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np, evaluate

model = AutoModelForSequenceClassification.from_pretrained(ckpt, num_labels=2)
accuracy = evaluate.load("accuracy")

def metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=preds, references=labels)

args = TrainingArguments(
    output_dir="sentiment_model",
    per_device_train_batch_size=16,
    eval_strategy="epoch",
    num_train_epochs=2,
    learning_rate=2e-5,
    logging_steps=50,
)
trainer = Trainer(model=model, args=args,
                  train_dataset=train_ds, eval_dataset=test_ds,
                  compute_metrics=metrics)
trainer.train()

## 4. Evaluate + confusion matrix

In [ ]:
preds = trainer.predict(test_ds)
y_hat = np.argmax(preds.predictions, axis=-1)
y     = np.array(test_ds["label"])

from sklearn.metrics import classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
print(classification_report(y, y_hat, target_names=["neg", "pos"]))
ConfusionMatrixDisplay.from_predictions(y, y_hat, display_labels=["neg", "pos"])
plt.show()

## 5. Predict on new reviews

In [ ]:
reviews = ["An absolute masterpiece - I was hooked from minute one.",
           "Two hours of my life I will never get back."]
enc = tok(reviews, return_tensors="pt", truncation=True, padding=True)
probs = torch.softmax(trainer.model(**enc).logits, dim=-1)
for r, p in zip(reviews, probs):
    print(f"{p.argmax().item()}  ({p.max().item():.2f})  {r}")

## Takeaways
- Fine-tune recipe: tokenize -> `AutoModelFor...WithClassificationHead` -> `Trainer`.
- Small LR (2e-5), few epochs - you are nudging, not relearning.
- Save with `trainer.save_model()` then reload anywhere with `from_pretrained`.